# Spatial Reasoning Without a Map

**DS4DH Practice Pack · Module 09 — Geospatial Analysis**

*Technique:* Ranking and grouping by geography when you have no boundary files

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/ds4dh-colab-pack/blob/main/notebooks/09a_ranking_grouping.ipynb)

Data: `merged_dataset.csv`, `top10_immigrant_penalty.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# This notebook reads the CSVs sitting next to it. In Colab you will be asked
# to upload them from the pack's data/ folder.
NEEDED = ['merged_dataset.csv', 'top10_immigrant_penalty.csv']

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('merged_dataset.csv')
df_top10 = pd.read_csv('top10_immigrant_penalty.csv')
CITIES = ['Montréal', 'Toronto', 'Edmonton', 'Vancouver']

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

There are no shapefiles in this dataset and no coordinates. You cannot draw a
choropleth.

That is less of a limitation than it sounds. Most geospatial *questions* are
about ranking, grouping and nesting — which places are worst, do they cluster in
particular cities, does the pattern hold within each city — and all three are
answerable from a geography column and an ID.

A map is a display format. Spatial reasoning is a way of asking.

In [ ]:
# One row per Census Subdivision.
#   • rows with no csd_code are CMA-level and Canada-level aggregates, not CSDs
#   • each CSD appears 3x (Immigrant / Non-immigrants / Total Immigrant Status)
# Keeping either would silently double- or triple-count places.
csd = df.dropna(subset=['csd_code'])
base = csd[(csd['immigrant_status'] == 'Total Immigrant Status')
           & (csd['cma'].isin(CITIES))].copy()

print(f'{len(df):>4} rows in the raw file')
print(f'{len(csd):>4} after dropping CMA/Canada aggregate rows')
print(f'{len(base):>4} CSDs in the four cities (one row each)')

In [ ]:
d = base.dropna(subset=['Total'])

print(f'{"Rank":>5}  {"Geography":<34}{"CMA":<11}{"STIR":>7}{"pop":>12}')
print('-' * 71)
for i, (_, r) in enumerate(d.nlargest(12, 'Total').iterrows(), 1):
    pop = f'{r["tot_pop"]:,.0f}' if pd.notna(r['tot_pop']) else 'n/a'
    print(f'{i:>5}  {r["geography_name"][:33]:<34}{r["cma"]:<11}'
          f'{r["Total"]:>7.1f}{pop:>12}')

## Grouping — does the ranking concentrate?

A league table of places becomes a spatial finding when you ask which city the
top entries belong to.

In [ ]:
TOP_N = 25
top = d.nlargest(TOP_N, 'Total')

share = pd.DataFrame({
    'in_top': top['cma'].value_counts(),
    'all_csds': d['cma'].value_counts(),
})
share['expected'] = share['all_csds'] / share['all_csds'].sum() * TOP_N
share['over_under'] = share['in_top'] - share['expected']
print(share.round(1).to_string())
print()
print('A city with more top-25 entries than its share of CSDs is')
print('over-represented among high-burden places. That is a spatial claim.')

In [ ]:
# Nesting — the same question asked within each city rather than across them.
print('Highest-burden CSD within each city:')
print()
for city in CITIES:
    g = d[d['cma'] == city]
    if not len(g):
        continue
    r = g.loc[g['Total'].idxmax()]
    print(f'{city:<12}{r["geography_name"][:34]:<36}{r["Total"]:>6.1f}'
          f'   (city median {g["Total"].median():.1f})')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
show = d.nlargest(15, 'Total').sort_values('Total')
colors = {c: col for c, col in zip(CITIES, ['#3DA5D9', '#E8663D', '#73BFB8', '#F2C14E'])}
ax.barh([n[:30] for n in show['geography_name']], show['Total'],
        color=[colors.get(c, '#999') for c in show['cma']])
ax.axvline(30, color='#888', ls=':', label='30% affordability line')
ax.set_xlabel('Total STIR (%)')
ax.set_title('Highest housing burden — bar colour is the CMA')
ax.legend()
plt.tight_layout()
plt.show()

### 🔧 Your turn 1

Change `TOP_N` to 50 and re-run the over/under table.

Does the same city stay over-represented? A concentration that only appears in
the top 10 is a claim about a handful of municipalities, not about a city.

## Ranking honestly

The same warning as in notebook 02d, in spatial form: the extremes of a ranking
are disproportionately small places, so a "worst places" map is partly a map of
where the census had the fewest households to average over.

In [ ]:
d_sized = d.dropna(subset=['tot_pop'])
q = d_sized['tot_pop'].quantile([0.25, 0.75])

small = d_sized[d_sized['tot_pop'] <= q.loc[0.25]]
large = d_sized[d_sized['tot_pop'] >= q.loc[0.75]]

print(f'{"":<22}{"n":>5}{"mean STIR":>12}{"sd of STIR":>13}')
print('-' * 52)
print(f'{"smallest quartile":<22}{len(small):>5}{small["Total"].mean():>12.1f}{small["Total"].std():>13.2f}')
print(f'{"largest quartile":<22}{len(large):>5}{large["Total"].mean():>12.1f}{large["Total"].std():>13.2f}')
print()
print('Small places vary more. Not because housing is more volatile there, but')
print('because a small denominator makes any average less stable.')

### 🔧 Your turn 2

Rebuild the top-15 chart using only CSDs above the median population.

How many of the original entries survive? Which version would you publish, and
what would the caption say?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** The over-representation usually persists at TOP_N = 50 but
weakens. That persistence is what turns "these municipalities are expensive" into
"this metropolitan area has a concentration of high-burden municipalities" — a
claim about a city rather than a list.

**Your turn 2.** Several small municipalities drop out and are replaced by larger
ones with slightly lower but far better-estimated burden. Publish the
size-filtered version, with a caption that states the filter: *"Census
subdivisions with population above the median (n = …). Smaller subdivisions are
excluded because their published ratios rest on very few households."* The
unfiltered chart is not wrong, but it invites a reader to conclude something the
data does not support.

</details>

## Where this stops

You can rank and group. You have not yet confronted the fact that the boundaries
themselves — which places count as one unit — change the answer. That is next.